# 重要性抽样

> 对应代码：`MathStatsCode/code_in_notes/Chap.MCMC/importance_sampling.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：以二元正态分布为工具密度做重要性抽样，大幅降低积分的模拟标准误。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.MCMC`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.MCMC")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

**重要性抽样计算积分示例**

清空、关闭分页、固定种子，准备重要性抽样实验。


In [ ]:
%%stata
clear all
set more off
set seed 8855


**设定参数**

用局部宏设定模拟次数 M=10000。


In [ ]:
%%stata
local M = 10000


**工具密度m(x)为独立的二元正态分布**

设定**工具密度**（importance density）m(x) 的参数：它是一个二元正态分布，两个维度的均值为 mu1=0.5、mu2=-0.1（正对目标函数的峰值位置），标准差为 σ1=√(1/180)、σ2=√(1/90)——这两个数值精心选择，使正态密度的指数部分 -（x-μ)²/(2σ²) 恰好还原目标函数中的 -90(x-0.5)² 与 -45(x+0.1)²（即 2σ² 的倒数）。


In [ ]:
%%stata
local mu1 = 0.5
local mu2 = -0.1
local sigma1 = sqrt(1/180)
local sigma2 = sqrt(1/90)


**从m(x)中抽样**

从工具分布抽样：`gen x1=rnormal(mu1,σ1)`、`gen x2=rnormal(mu2,σ2)`——`rnormal(均值,标准差)` 的第二个参数是标准差。与简单 Monte Carlo 从均匀分布抽样不同，这里把样本“集中”在函数值大的地方，效率更高。


In [ ]:
%%stata
set obs `M'
gen x1 = rnormal(`mu1', `sigma1')
gen x2 = rnormal(`mu2', `sigma2')


**目标函数h1_star(x)=4*h1(x)**

`gen h1_star = 4*0.5*exp(...)`：把目标积分写成对**原密度**的期望——h1_star=4·h1(x)，其中系数4来自原密度 pai(x)=1/4·I(x∈[-1,1]²) 的“面积”倒数。


In [ ]:
%%stata
gen h1_star = 4*0.5*exp(-90*(x1-0.5)^2 - 45*(x2+0.1)^2)


**原密度pai(x)=1/4*I(x\in[-1,1]^2)**

`gen pai = 1/4*(x1>=-1 & x1<=1 & x2>=-1 & x2<=1)`：原密度（定义在矩形上的均匀分布密度，区域外为0）。这里借用了 Stata 的逻辑值运算：条件成立返回1、不成立返回0。


In [ ]:
%%stata
gen pai = 1/4*(x1>=-1 & x1<=1 & x2>=-1 & x2<=1)


**工具密度m(x)**

`gen m = 1/(2πσ1σ2)·exp(-(x1-μ1)²/(2σ1²)-(x2-μ2)²/(2σ2²))`：二元正态**工具密度的密度值**，即重要性抽样公式中的分母 m(x)。注意 `_pi` 是 Stata 内置常数 π。


In [ ]:
%%stata
gen m = 1/(2*_pi*`sigma1'*`sigma2') ///
	*exp(-(x1-`mu1')^2/(2*`sigma1'^2) - (x2-`mu2')^2/(2*`sigma2'^2))


**计算积分**

计算重要性抽样估计：`gen H = h1_star*pai/m`，则积分估计 Î=(1/M)ΣH(x_i)（`su H` 取均值）、标准误=s_H/√M。与简单 Monte Carlo 的结果对比可见：**标准误数量级下降**（因为是精心选定的工具密度）。权重 pai/m 的含义是：在工具分布下抽样时，用似然比把“抽样概率”校正回原分布的期望。


In [ ]:
%%stata
gen H = h1_star*pai/m
qui: su H
local integral = r(mean)
local se = r(sd)/sqrt(`M')
di "Integral = " `integral'
di "s.e. of Integral = " `se'
di "95% C.I.: " `integral'-1.96*`se' " ~ " `integral'+1.96*`se'
